# 06 · RAGAS: grading the answers, not just the retrieval
Inspects `ragtraining.eval.ragas_lite` / `run_ragas` (LOGBOOK §7.3). Judge = **Haiku 5.5**, answers = **Opus 5.5**. Reads stored results: **$0**.
- **faithfulness**: share of the answer's claims supported by the sources → *does it make things up?*
- **answer relevancy**: does it answer *this* question? (questions regenerated from the answer vs. the real one)
- **context precision**: are the useful sources ranked high?
- **context recall**: did retrieval bring everything the reference answer needs?

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
M = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
old = {p: json.load(open(f'../data/ragas_{p}.json')) for p in ['baseline', 'prod']}
new = {p: json.load(open(f'../data/ragas_{p}_rejudged.json')) for p in ['baseline', 'prod']}
pd.DataFrame({f'{p} · {w}': d[p]['summary'] for w, d in [('old judge', old), ('fixed judge', new)] for p in d}).loc[M]

## 1. P-33: the judge must see what the tutor saw
The tutor ends with *"Review: Lecture 3 slides p.1-4"*. Those page numbers come from source **metadata**. The old judge only
got the chunk **text**, so it marked every such pointer as unsupported. Same answers, same sources, the judge just sees the headers now:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, p in zip(ax, ['baseline', 'prod']):
    o = [r['faithfulness'] for r in old[p]['rows']]; n = [r['faithfulness'] for r in new[p]['rows']]
    x = np.arange(len(o)); a.bar(x - .2, o, .4, label='old judge'); a.bar(x + .2, n, .4, label='fixed judge')
    a.set(title=f'{p}: faithfulness per question', xlabel='question'); a.legend(loc='lower left')
plt.tight_layout(); plt.show()
for p in new:
    n_old = sum(len(r['unsupported']) for r in old[p]['rows']); n_new = sum(len(r['unsupported']) for r in new[p]['rows'])
    print(f'{p:9s} unsupported claims: {n_old} -> {n_new}')

## 2. What is still "unsupported"?
Mostly the tutor's own **inferences** and mnemonics. They're reasonable, but not stated in the sources. G1 says *only from the
material*, so the judge is right to flag them.

In [ ]:
pd.DataFrame([(p, u) for p in new for r in new[p]['rows'] for u in r['unsupported']], columns=['pipeline', 'claim']).head(15)

## 3. Judge noise (P-36)
Context precision and recall don't depend on the answer, and the sources were *identical* in both runs. Any change is the
judge's own randomness. Rule of thumb: differences under ~0.03 on 20 questions are noise.

In [ ]:
pd.DataFrame({p: {m: round(new[p]['summary'][m] - old[p]['summary'][m], 3) for m in ['context_precision', 'context_recall']}
              for p in new}).rename_axis('change with SAME sources')

## 4. Final run (S3, D24): local engine on all 70 questions vs. Claude engine on Max's 12
Current concise tutor, final test set (58 synthetic after the spot-check + Max's 12). Local answers cost $0, only the judge is paid.

In [ ]:
fin = {'local · all 70': json.load(open('../data/ragas_local_all.json')),
       'claude · Max 12': json.load(open('../data/ragas_prod_manual.json'))}
tab = {k: {m: v['summary'][m] for m in M} | {'n': v['summary']['n'], 'citation problems': v['summary']['citation_problems']}
       for k, v in fin.items()}
loc = fin['local · all 70']['summary']['by_source']
tab |= {f'local · {g}': {m: loc[g][m] for m in M} | {'n': loc[g]['n']} for g in loc}
pd.DataFrame(tab).T

**Reading:** both engines are faithful (0.96 vs 0.99 on the same 12). The big gap is **context precision** (0.75 vs 0.95):
the Claude *reranker* puts more relevant chunks in the top 8. Answer relevancy is low on Max's vague questions for both
engines ("what is DS?"), which is a property of the metric, not a wrong answer.

## 5. P-53 → D25: a check that only logs is a measurement, not a guardrail
8/70 local answers had no inline `[S#]`. In 6 of them qwen3 named the sources only in the final *"Review: S1, p.4"* line.
Fix: if an answer has no citations, the tutor asks **once more** with a reminder (`tutor.CITE_REMINDER`). `run_ragas patch`
re-answered only those 8 rows on the **same evidence** and re-judged them (Haiku, ≈ $0.01):

In [ ]:
p = json.load(open('../data/ragas_local_all_patched.json'))
before = {r['question']: r for r in fin['local · all 70']['rows']}
pd.DataFrame([dict(question=r['question'][:55], cited_before='[S' in r['old_answer'], cited_after='[S' in (r['answer'] or ''),
                   faith_before=before[r['question']]['faithfulness'], faith_after=r['faithfulness'])
              for r in p['rows'] if r.get('patched')])

## 6. P11 → D25: the regression gate
A threshold turns a score into pass/fail. Every bar sits at least one judge-noise step (±0.03) below today's score: a real
drop fails, noise doesn't. Answer relevancy is a *soft* bar (warns only). Run it after any component change:
`python -m ragtraining.eval.run_ragas gate <name>` (exit code 1 on FAIL).

In [ ]:
from ragtraining.eval.run_ragas import gate
for name in ['local_all', 'local_all_patched']:
    print(f'--- {name}'); gate(name)